# Music Annotation Project V8.8B
## Notebook 53R2: Exact Notebook 51R2-Compatible Operational M5_REMOVE Builder

**Runtime:** CPU. Select **Runtime → Run all**.

Notebook 53 consumes the frozen Notebook 52 policy identity and builds operational M5_REMOVE page artefacts from an observed page, an authoritative intervention mask/M3 mask, and SAFE_REMOVE decisions produced by the frozen Notebook 50 policy.

### Fixed contract
- `M5_REMOVE = M3 & ~SAFE_REMOVE`.
- SAFE_PROTECT is not enabled.
- No retraining, recalibration or threshold movement.
- Clean truth is prohibited from inference.
- SAFE_REMOVE must be a subset of M3.
- Unsupported, missing, invalid or uncertain coordinates retain exact M3 treatment.
- Pixels outside the authoritative mask remain byte-for-byte equal to the observed page.
- This notebook does not replace Notebook 09 by itself. The registered replay must exactly reproduce Notebook 51R2 M5/R2 Telea corrected pages before Notebook 54 is rerun.

### Two input routes
1. **Frozen regression route:** uses Notebook 50's persisted all-M3 OOF coordinate roster and SAFE_REMOVE decisions for pages in the registered V8.7 roster; Notebook 45 is not an execution dependency. This is the mandatory first execution route.
2. **Operational feature route:** accepts a page-level predictor table produced by the exact frozen Notebook 50 observable-feature builder. This route stops unless a compatible deployable REMOVE scorer can be resolved and replay-verified.

Binary model, calibration, parquet and NPZ artefacts remain in Google Drive and are validated in place.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy pyarrow joblib scikit-learn opencv-python-headless


## 1. Configuration and output identity


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, zipfile, sys, platform, os, re, traceback
import numpy as np
import pandas as pd
import cv2, joblib

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
N52_ROOT=ROOT/'m5_remove_policy_freeze_v8_8a'
N51_ROOT=ROOT/'observed_print_classifier_v8_7_notebook51'
OUTPUT_ROOT=ROOT/'m5_remove_operational_builder_v8_8b'

# Optional exact run pins. Leave None to use latest complete runs.
NOTEBOOK52_RUN_OVERRIDE=None
NOTEBOOK50_RUN_OVERRIDE=None
NOTEBOOK51_RUN_OVERRIDE=None

# Mandatory first execution is registered V8.7 replay.
INPUT_MODE='REGISTERED_REPLAY'  # REGISTERED_REPLAY or OPERATIONAL_FEATURE_TABLES

# Operational inputs, used only when INPUT_MODE == OPERATIONAL_FEATURE_TABLES.
OPERATIONAL_ROOT=ROOT/'v8_8_operational_inputs'
OBSERVED_DIR=OPERATIONAL_ROOT/'observed'
M3_MASK_DIR=OPERATIONAL_ROOT/'m3_masks'
FEATURE_DIR=OPERATIONAL_ROOT/'features'
SOURCE_MANIFEST=OPERATIONAL_ROOT/'page_manifest.csv'

# Reconstruction/output contract. Telea is the initial frozen operational method;
# Notebook 54 must compare it against the registered Notebook 51 arm.
RECONSTRUCTION_METHOD='telea'
INPAINT_RADIUS=3
CROP_MARGIN=48  # retained for historical metadata; not used by exact R2 composition

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook53_v8_8b'
OUT=OUTPUT_ROOT/RUN
INPUTS=OUT/'inputs'; TABLES=OUT/'tables'; MASKS=OUT/'masks'; PAGES=OUT/'pages'; AUDIT=OUT/'audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,TABLES,MASKS,PAGES,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
print('Mode:',INPUT_MODE); print('Output:',OUT)


## 2. Helpers and fail-closed source resolution


In [ ]:
def require(condition,message):
    if not bool(condition): raise RuntimeError(message)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def read_json(path): return json.loads(Path(path).read_text())

def atomic_json(path,value):
    tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def image_map(folder):
    return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}

def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def resolve_unique(raw,roots=()):
    p=Path(str(raw)); candidates=[p]+[Path(r)/p for r in roots]+[Path(r)/p.name for r in roots]
    unique={}
    for x in candidates:
        if x.exists(): unique[str(x.resolve())]=x.resolve()
    hits=list(unique.values())
    require(len(hits)==1,f'Expected one file for {raw}; found {len(hits)} unique files: {hits}')
    return hits[0]

def binary(a): return np.asarray(a)>0


## 3. Resolve and verify the passed Notebook 52 policy


In [ ]:
N52_REQUIRED=['notebook52_verdict.json','policy_bundle/policy_manifest.json','tables/m5_remove_golden_master_semantic_hashes.csv','tables/notebook50_full_artifact_inventory.csv']
N52=Path(NOTEBOOK52_RUN_OVERRIDE) if NOTEBOOK52_RUN_OVERRIDE else latest_complete(N52_ROOT,N52_REQUIRED)
require(N52 is not None and N52.exists(),'No complete Notebook 52 V8.8A run found.')
verdict52=read_json(N52/'notebook52_verdict.json'); policy=read_json(N52/'policy_bundle'/'policy_manifest.json')
require(verdict52.get('verdict')=='PASS_M5_REMOVE_FREEZE_READY_FOR_OPERATIONAL_BUILDER',f'Notebook 52 did not pass: {verdict52}')
require(policy.get('policy')=='M5_REMOVE','Notebook 52 policy is not M5_REMOVE.')
require(policy.get('safe_protect_runtime_enabled') is False,'SAFE_PROTECT must remain disabled.')
require(policy.get('retraining_performed') is False and policy.get('recalibration_performed') is False and policy.get('threshold_changed') is False,'Notebook 52 policy mutation detected.')
EXPECTED_POLICY_ID=policy['policy_id']
N50=Path(NOTEBOOK50_RUN_OVERRIDE) if NOTEBOOK50_RUN_OVERRIDE else Path(policy['authoritative_notebook50_run'])
require(N50.exists(),f'Authoritative Notebook 50 run missing: {N50}')
print('Notebook 52:',N52); print('Notebook 50:',N50); print('Policy ID:',EXPECTED_POLICY_ID)


## 3A. Resolve the exact Notebook 51R2 reconstruction lineage


In [ ]:
N51_REQUIRED=['source_record.json','notebook51_verdict.json','01_pre_registration.json','03_candidates','05_tables/mask_arm_inventory.csv']
N51=Path(NOTEBOOK51_RUN_OVERRIDE) if NOTEBOOK51_RUN_OVERRIDE else latest_complete(N51_ROOT,N51_REQUIRED)
require(N51 is not None and N51.exists(),'No complete Notebook 51R2 run found. Set NOTEBOOK51_RUN_OVERRIDE.')
source51=read_json(N51/'source_record.json')
verdict51=read_json(N51/'notebook51_verdict.json')
require(verdict51.get('valid_run') is True,'Notebook 51R2 valid_run is not True.')
require(verdict51.get('m5_policy')=='REMOVE-only','Notebook 51R2 is not REMOVE-only.')
require(source51.get('notebook50') is not None,'Notebook 51R2 source record lacks notebook50.')
require(Path(source51['notebook50']).resolve()==N50.resolve(),'Notebook 51R2 and Notebook 52 identify different Notebook 50 runs.')
N45=Path(source51['notebook45']); N46=Path(source51['notebook46'])
require(N45.exists() and N46.exists(),f'Notebook 51R2 lineage paths missing: N45={N45}, N46={N46}')
for p in [N45/'mask_lock.json',N45/'source_record.json',N45/'tables/region_manifest.parquet',N46/'whole_page_candidates',N46/'source_record.json']:
    require(p.exists(),f'Missing frozen reconstruction-lineage artefact: {p}')
source45=read_json(N45/'source_record.json')
SOURCE_V82=Path(source45['source_v82'])
require(SOURCE_V82.exists(),f'Notebook 45 source_v82 missing: {SOURCE_V82}')
regions=pd.read_parquet(N45/'tables/region_manifest.parquet')
need={'page','source_group','region_id','y0','y1','x0','x1'}
require(need.issubset(regions.columns),f'Notebook 45 region manifest missing {sorted(need-set(regions.columns))}')
regions['page']=regions.page.astype(str); regions['source_group']=regions.source_group.astype(str)
require(not regions[['page','region_id']].duplicated().any(),'Duplicate Notebook 45 page/region identity.')
print('Notebook 51R2:',N51)
print('Notebook 45:',N45)
print('Notebook 46R6:',N46)
print('Regions:',len(regions),'Pages:',regions.page.nunique())


## 4. Resolve Notebook 50 schema, predictions and decision encoding


In [ ]:
schema=read_json(N50/'inputs'/'predictor_schema.json')
PREDICTORS=schema.get('ordered_predictors',schema.get('predictors',schema.get('ordered_features',[])))
require(len(PREDICTORS)>0,'Frozen ordered predictor schema is empty.')

pred_path=N50/'tables'/'all_m3_oof_predictions_and_decisions.parquet'
require(pred_path.exists(),f'Missing Notebook 50 decision table: {pred_path}')
pred=pd.read_parquet(pred_path)
required={'page','y','x','safe_remove'}
require(required.issubset(pred.columns),f'Notebook 50 prediction table missing: {sorted(required-set(pred.columns))}')
pred['page']=pred.page.astype(str); pred['y']=pd.to_numeric(pred.y,errors='raise').astype(int); pred['x']=pd.to_numeric(pred.x,errors='raise').astype(int)
pred['safe_remove']=pred.safe_remove.astype(bool)
require(not pred.duplicated(['page','y','x']).any(),'Duplicate Notebook 50 page/y/x decision rows.')

map_inventory=pd.read_csv(N50/'tables'/'m5_map_inventory.csv'); map_inventory['page']=map_inventory.page.astype(str)
require({'page','file'}.issubset(map_inventory.columns),'Notebook 50 m5_map_inventory lacks page/file.')

# Infer and verify m5_decision_map encoding from pages
# that actually contain SAFE_REMOVE assertions.
#
# Pages with zero SAFE_REMOVE rows are valid M3-fallback pages
# and must not be used to infer the SAFE_REMOVE integer code.

encoding_rows = []

for page, g in pred.groupby('page'):

    inv = map_inventory[
        map_inventory.page == page
    ]

    require(
        len(inv) == 1,
        f'{page}: expected one map inventory row, '
        f'found {len(inv)}'
    )

    p = resolve_unique(
        inv.iloc[0]['file'],
        [N50, N50/'maps', N50/'masks']
    )

    with np.load(p, allow_pickle=False) as z:

        require(
            {'m5_decision_map', 'reason_map'}.issubset(z.files),
            f'{p}: missing decision arrays; keys={z.files}'
        )

        dm = np.asarray(z['m5_decision_map'])
        rm = np.asarray(z['reason_map'])

    require(
        dm.shape == rm.shape and dm.ndim == 2,
        f'{page}: decision/reason shape failure.'
    )

    require(
        (g.y >= 0).all()
        and (g.x >= 0).all()
        and (g.y < dm.shape[0]).all()
        and (g.x < dm.shape[1]).all(),
        f'{page}: prediction coordinate outside saved '
        f'decision canvas.'
    )

    values = dm[
        g.y.to_numpy(),
        g.x.to_numpy()
    ]

    sr_mask = g.safe_remove.to_numpy(dtype=bool)

    safe_remove_rows = int(sr_mask.sum())

    # -----------------------------------------------
    # Page contains one or more SAFE_REMOVE decisions
    # -----------------------------------------------

    if safe_remove_rows > 0:

        srvals = values[sr_mask]
        unique_sr = np.unique(srvals)

        require(
            len(unique_sr) == 1,
            f'{page}: SAFE_REMOVE rows map to multiple '
            f'decision codes {unique_sr}'
        )

        safe_remove_code = int(unique_sr[0])

    # -----------------------------------------------
    # Zero SAFE_REMOVE is a legitimate page outcome.
    # There is no code to infer from this page.
    # -----------------------------------------------

    else:

        safe_remove_code = None

    encoding_rows.append({
        'page': page,

        'safe_remove_rows':
            safe_remove_rows,

        'safe_remove_code':
            safe_remove_code,

        'all_decision_values':
            '|'.join(
                map(str, np.unique(dm).tolist())
            ),

        'decision_shape':
            str(dm.shape),

        'zero_safe_remove_page':
            bool(safe_remove_rows == 0),
    })


encoding = pd.DataFrame(encoding_rows)

# ---------------------------------------------------
# Infer the global SAFE_REMOVE code only from pages
# where SAFE_REMOVE was actually asserted.
# ---------------------------------------------------

observed_codes = (
    encoding.loc[
        encoding.safe_remove_rows > 0,
        'safe_remove_code'
    ]
    .dropna()
    .astype(int)
    .unique()
)

require(
    len(observed_codes) == 1,
    'Could not identify one consistent Notebook 50 '
    f'SAFE_REMOVE code. Observed codes: {observed_codes}'
)

SAFE_REMOVE_CODE = int(observed_codes[0])

# Every page which DID contain SAFE_REMOVE must agree
# with that global code.
require(
    (
        encoding.loc[
            encoding.safe_remove_rows > 0,
            'safe_remove_code'
        ].astype(int)
        == SAFE_REMOVE_CODE
    ).all(),
    'Notebook 50 SAFE_REMOVE encoding is inconsistent '
    'between pages.'
)

encoding.to_csv(
    AUDIT/'decision_encoding_audit.csv',
    index=False
)

print('Predictors:', len(PREDICTORS))
print('SAFE_REMOVE decision code:', SAFE_REMOVE_CODE)
print(
    'Pages with SAFE_REMOVE:',
    int((encoding.safe_remove_rows > 0).sum())
)
print(
    'Pages with zero SAFE_REMOVE:',
    int((encoding.safe_remove_rows == 0).sum())
)

display(
    encoding[
        [
            'page',
            'safe_remove_rows',
            'safe_remove_code',
            'all_decision_values',
            'zero_safe_remove_page',
        ]
    ]
)


## 5. Resolve page inputs for the selected route


In [ ]:
def find_page_image(page,folders):
    for folder in folders:
        folder=Path(folder)
        if not folder.exists(): continue
        hits={str(p.resolve()):p.resolve() for p in folder.glob(page+'.*') if p.is_file() and p.suffix.lower() in SUPPORTED}
        found=list(hits.values())
        require(len(found)<=1,f'{page}: multiple images in {folder}: {found}')
        if found: return found[0]
    raise RuntimeError(f'{page}: no supported image found in {folders}')


def discover_reference_page_inputs(page):
    observed=SOURCE_V82/'whole_pages'/page/'observed.png'
    require(observed.exists(),f'{page}: missing Notebook 51R2-lineage observed page: {observed}')
    image=cv2.imread(str(observed),cv2.IMREAD_GRAYSCALE)
    require(image is not None,f'{page}: unreadable observed image {observed}')
    q=pred[pred.page.astype(str)==str(page)].copy()
    require(len(q)>0,f'{page}: no Notebook 50 M3 rows')
    require(not q.duplicated(['y','x']).any(),f'{page}: duplicate Notebook 50 y/x rows')
    ys=q.y.to_numpy(int); xs=q.x.to_numpy(int); h,w=image.shape
    require((ys>=0).all() and (xs>=0).all() and (ys<h).all() and (xs<w).all(),f'{page}: Notebook 50 coordinates outside {image.shape}')
    m3=np.zeros(image.shape,bool); m3[ys,xs]=True
    require(int(m3.sum())==len(q),f'{page}: M3 reconstruction mismatch')
    d=INPUTS/'reconstructed_m3'; d.mkdir(parents=True,exist_ok=True)
    m3_path=d/f'{page}_M3.png'
    require(cv2.imwrite(str(m3_path),m3.astype(np.uint8)*255),f'{page}: could not write M3')
    return observed,m3_path,m3

if INPUT_MODE=='REGISTERED_REPLAY':
    roster=sorted(pred.page.unique())
    source_rows=[]
    for page in roster:
        obs,m3,_=discover_reference_page_inputs(page)
        source_rows.append({'page':page,'observed_path':str(obs),'m3_path':str(m3),'feature_path':str(pred_path),'source_group':str(pred.loc[pred.page==page,'source_group'].iloc[0]) if 'source_group' in pred.columns else '', 'm3_source':'NOTEBOOK50_ALL_M3_COORDINATES','reconstruction_lineage':'NOTEBOOK51R2_SOURCE_RECORD'})
    sources=pd.DataFrame(source_rows)
elif INPUT_MODE=='OPERATIONAL_FEATURE_TABLES':
    require(SOURCE_MANIFEST.exists(),f'Missing operational page manifest: {SOURCE_MANIFEST}')
    sources=pd.read_csv(SOURCE_MANIFEST,dtype=str)
    require({'page','source_group'}.issubset(sources.columns),'Operational manifest needs page/source_group.')
    sources['page']=sources.page.astype(str)
    rows=[]
    for _,r in sources.iterrows():
        page=r.page; obs=find_page_image(page,[OBSERVED_DIR]); m3=find_page_image(page,[M3_MASK_DIR]); fp=FEATURE_DIR/f'{page}.parquet'
        require(fp.exists(),f'{page}: missing operational feature table {fp}')
        rows.append({'page':page,'source_group':r.source_group,'observed_path':str(obs),'m3_path':str(m3),'feature_path':str(fp),'reconstruction_lineage':'FUTURE_OPERATIONAL'})
    sources=pd.DataFrame(rows)
else: raise ValueError(INPUT_MODE)
require(not sources.page.duplicated().any(),'Input pages must be unique.')
sources.to_csv(INPUTS/'resolved_page_inputs.csv',index=False)
print('Pages:',len(sources))
print('Registered replay uses Notebook 51R2 exact Notebook 45/46 source lineage.')


## 6. Freeze-compatible REMOVE scoring

The registered replay route consumes Notebook 50's exact persisted OOF decisions. The operational feature route requires a separately serialised deployable scorer whose schema and policy identity match Notebook 52. Fold OOF decisions are not silently converted into an unseen-page ensemble.


In [ ]:
def find_deployable_scorer():
    candidates=[]
    for p in N50.rglob('*.joblib'):
        name=p.name.lower()
        if ('deploy' in name or 'final' in name or 'full' in name) and ('remove' in name or 'm5' in name): candidates.append(p.resolve())
    candidates=list(dict.fromkeys(candidates))
    require(len(candidates)==1,'Operational feature mode requires exactly one explicitly deployable full-development REMOVE scorer. '
            f'Found {candidates}. Do not substitute OOF fold models or retrain in Notebook 53.')
    return candidates[0]

def decisions_for_page(page,feature_path):
    if INPUT_MODE=='REGISTERED_REPLAY':
        q=pred[pred.page==page].copy()
        return q[['page','y','x','safe_remove']+[c for c in ['p_remove','remove_probability','m5_reason'] if c in q.columns]]
    features=pd.read_parquet(feature_path)
    require({'page','y','x'}.issubset(features.columns),f'{page}: feature table lacks page/y/x.')
    require(not features.duplicated(['page','y','x']).any(),f'{page}: duplicate feature coordinates.')
    missing=[c for c in PREDICTORS if c not in features.columns]
    require(not missing,f'{page}: missing frozen predictors: {missing}')
    X=features.loc[:,PREDICTORS].apply(pd.to_numeric,errors='coerce')
    require(not np.isinf(X.to_numpy(float)).any(),f'{page}: infinite predictor value.')
    scorer_path=find_deployable_scorer(); scorer=joblib.load(scorer_path)
    require(hasattr(scorer,'predict_proba') or hasattr(scorer,'predict'),f'{scorer_path}: scorer has no prediction method.')
    if hasattr(scorer,'predict_proba'):
        prob=np.asarray(scorer.predict_proba(X))[:,1]
        # Threshold must be discoverable from Notebook 50 verdict/control records.
        threshold=None
        for obj in [read_json(N50/'notebook50_verdict.json'),read_json(N50/'selected_models.json')]:
            for key in ['remove_threshold','safe_remove_threshold','threshold_remove']:
                if key in obj: threshold=float(obj[key])
        require(threshold is not None,'Frozen REMOVE threshold could not be resolved from Notebook 50 records.')
        selected=prob>=threshold
    else:
        selected=np.asarray(scorer.predict(X)).astype(bool); prob=np.full(len(selected),np.nan)
    return pd.DataFrame({'page':page,'y':features.y.astype(int),'x':features.x.astype(int),'safe_remove':selected,'p_remove':prob})


## 7. Canonical M5_REMOVE masks and correction outputs


In [ ]:
def reconstruct_classical_exact(obs,mask):
    m=(mask>0).astype(np.uint8)*255
    if not m.any(): return obs.copy()
    require(RECONSTRUCTION_METHOD=='telea','Exact Notebook 51R2 path requires Telea.')
    cand=cv2.inpaint(obs,m,3,cv2.INPAINT_TELEA)
    out=obs.copy(); out[m>0]=cand[m>0]
    require(np.array_equal(out[m==0],obs[m==0]),'Telea outside-regional-mask drift.')
    return out


def load_exact_reconstruction_context(page,observed,m3):
    base_path=N46/'whole_page_candidates'/page/'v7_8.png'
    protected_path=N46/'whole_page_candidates'/page/'protected_print.png'
    if not base_path.exists(): base_path=SOURCE_V82/'whole_pages'/page/'v7_8.png'
    base=cv2.imread(str(base_path),0); protected=cv2.imread(str(protected_path),0)
    require(base is not None and protected is not None,f'{page}: missing frozen V7.8 base/protected-print source.')
    require(base.shape==protected.shape==observed.shape==m3.shape,f'{page}: frozen reconstruction context shape mismatch.')
    pg=regions[regions.page==page].copy()
    require(len(pg)>0,f'{page}: no Notebook 45 regions.')
    regional=[]
    for _,r in pg.iterrows():
        mp=N45/'masks'/page/str(r.region_id).replace(':','_')/'M3.png'
        rm=cv2.imread(str(mp),0)
        require(rm is not None and rm.shape==observed.shape,f'{page}: invalid regional M3 {mp}')
        regional.append((str(r.region_id),rm>0,mp))
    union=np.zeros(m3.shape,bool)
    for _,rm,_ in regional: union|=rm
    require(np.array_equal(union,m3),f'{page}: Notebook 45 regional M3 union differs from Notebook 50 M3.')
    return base,protected>0,regional,base_path,protected_path


def build_output_exact(page,observed,m3,safe_remove):
    require(not np.any(safe_remove & ~m3),f'{page}: SAFE_REMOVE outside M3.')
    m5=m3 & ~safe_remove
    base,protected,regional_masks,base_path,protected_path=load_exact_reconstruction_context(page,observed,m3)
    candidate=base.copy(); candidate[protected]=observed[protected]
    regional_union=np.zeros(m3.shape,bool)
    region_rows=[]
    for region_id,region_m3,mp in regional_masks:
        regional=m5 & region_m3
        regional_union|=regional
        if regional.any():
            recon=reconstruct_classical_exact(observed,regional)
            candidate[regional]=recon[regional]
        region_rows.append({'page':page,'region_id':region_id,'regional_m3_pixels':int(region_m3.sum()),'regional_m5_pixels':int(regional.sum()),'m3_path':str(mp)})
    require(np.array_equal(regional_union,m5),f'{page}: regional M5 union differs from page M5.')
    auth=np.asarray(m3|safe_remove)  # M3 is authoritative for this builder's changed-region check
    require(np.array_equal(candidate[protected & ~m5],observed[protected & ~m5]),f'{page}: protected non-residual drift.')
    return m5,candidate,base,protected,region_rows,base_path,protected_path


## 8. Execute every page and persist separate audit artefacts


In [ ]:
page_rows=[]; decision_rows=[]; region_audit_rows=[]

# ------------------------------------------------------------
# Resolve Notebook 51R2 golden-master candidate inventory
# ------------------------------------------------------------

n51_inventory_path = (
    N51 /
    '04_candidate_lock' /
    'candidate_inventory.csv'
)

require(
    n51_inventory_path.exists(),
    f'Missing Notebook51R2 candidate inventory: '
    f'{n51_inventory_path}'
)

n51_candidates = pd.read_csv(
    n51_inventory_path,
    dtype={'page': str}
)

required_n51_cols = {
    'page',
    'arm',
    'restorer',
    'candidate',
    'sha256',
    'array_sha256',
}

require(
    required_n51_cols.issubset(
        n51_candidates.columns
    ),
    'Notebook51R2 candidate inventory missing columns: '
    f'{sorted(required_n51_cols - set(n51_candidates.columns))}'
)

n51_candidates['page'] = (
    n51_candidates.page.astype(str)
)

print(
    'Notebook51R2 candidate inventory:',
    n51_inventory_path
)

print(
    'M5 / R2_telea references:',
    len(
        n51_candidates[
            (n51_candidates.arm == 'M5_REMOVE')
            &
            (n51_candidates.restorer == 'R2_telea')
        ]
    )
)

for _,src in sources.iterrows():
    page=str(src.page); observed=cv2.imread(str(src.observed_path),0); m3_image=cv2.imread(str(src.m3_path),0)
    require(observed is not None and m3_image is not None,f'{page}: unreadable observed/M3 image.')
    m3=m3_image>0; require(observed.shape==m3.shape,f'{page}: observed/M3 shape mismatch.')
    decisions=decisions_for_page(page,Path(src.feature_path)); decisions['page']=decisions.page.astype(str)
    require((decisions.y>=0).all() and (decisions.x>=0).all() and (decisions.y<observed.shape[0]).all() and (decisions.x<observed.shape[1]).all(),f'{page}: decision coordinate outside page.')
    safe_remove=np.zeros(observed.shape,bool); q=decisions[decisions.safe_remove.astype(bool)]; safe_remove[q.y.to_numpy(int),q.x.to_numpy(int)]=True
    require(not np.any(safe_remove & ~m3),f'{page}: selected SAFE_REMOVE outside M3.')
    m5,out,base,protected,region_rows,base_path,protected_path=build_output_exact(page,observed,m3,safe_remove)
    region_audit_rows.extend(region_rows)
    d=PAGES/page; d.mkdir(parents=True,exist_ok=False)
    cv2.imwrite(str(d/'observed.png'),observed); cv2.imwrite(str(d/'m3.png'),m3.astype(np.uint8)*255); cv2.imwrite(str(d/'safe_remove.png'),safe_remove.astype(np.uint8)*255); cv2.imwrite(str(d/'m5_remove.png'),m5.astype(np.uint8)*255); cv2.imwrite(str(d/'frozen_v7_8_base.png'),base); cv2.imwrite(str(d/'protected_print.png'),protected.astype(np.uint8)*255); cv2.imwrite(str(d/'corrected.png'),out)
    diff=np.zeros_like(observed); diff[out!=observed]=255; cv2.imwrite(str(d/'difference_from_observed.png'),diff)
    np.savez_compressed(MASKS/f'{page}_m5_remove_operational.npz',M3=m3.astype(np.uint8),SAFE_REMOVE=safe_remove.astype(np.uint8),M5_REMOVE=m5.astype(np.uint8),PROTECTED_PRINT=protected.astype(np.uint8))
    decisions.to_parquet(TABLES/f'{page}_safe_remove_decisions.parquet',index=False)
    for _,r in decisions.iterrows(): decision_rows.append({'page':page,'y':int(r.y),'x':int(r.x),'safe_remove':bool(r.safe_remove),'p_remove':r.get('p_remove',np.nan)})

    # ------------------------------------------------------------
    # Exact Notebook51R2 M5 / R2_telea golden-master comparison
    # ------------------------------------------------------------

    ref_row = n51_candidates[
        (n51_candidates.page == page)
        &
        (n51_candidates.arm == 'M5_REMOVE')
        &
        (n51_candidates.restorer == 'R2_telea')
    ]

    require(
        len(ref_row) == 1,
        f'{page}: expected exactly one Notebook51R2 '
        f'M5/R2_telea candidate, found {len(ref_row)}.'
    )

    ref = Path(
        ref_row.iloc[0]['candidate']
    )

    # If the original absolute path has moved, recover only from
    # the resolved Notebook51R2 candidate directory.
    if not ref.exists():

        fallback = (
            N51 /
            '03_candidates' /
            page /
            ref.name
        )

        require(
            fallback.exists(),
            f'{page}: Notebook51R2 M5/R2_telea '
            f'candidate missing.\n'
            f'Inventory path: {ref}\n'
            f'Fallback path: {fallback}'
        )

        ref = fallback


    # Verify the golden-master FILE itself before comparing arrays.
    expected_file_sha = str(
        ref_row.iloc[0]['sha256']
    )

    actual_file_sha = sha256_file(ref)

    require(
        actual_file_sha == expected_file_sha,
        f'{page}: Notebook51R2 candidate file SHA-256 '
        f'mismatch.\n'
        f'Expected: {expected_file_sha}\n'
        f'Actual:   {actual_file_sha}\n'
        f'File: {ref}'
    )


    refim = cv2.imread(
        str(ref),
        cv2.IMREAD_GRAYSCALE
    )

    require(
        refim is not None,
        f'{page}: Notebook51R2 reference unreadable: {ref}'
    )

    require(
        refim.shape == out.shape,
        f'{page}: Notebook51R2 reference shape '
        f'{refim.shape} != operational output shape '
        f'{out.shape}.'
    )


    # Verify semantic array identity against Notebook51's own lock.
    expected_array_sha = str(
        ref_row.iloc[0]['array_sha256']
    )

    actual_array_sha = array_hash(refim)

    require(
        actual_array_sha == expected_array_sha,
        f'{page}: Notebook51R2 reference array SHA-256 '
        f'does not match candidate lock.\n'
        f'Expected: {expected_array_sha}\n'
        f'Actual:   {actual_array_sha}'
    )


    # Now, and only now, perform the actual equivalence test.
    ref_diff = int(
        np.sum(out != refim)
    )

    exact_reference = (
        ref_diff == 0
    )

    reference_available = True



    page_rows.append({'page':page,'source_group':src.source_group,'observed_path':src.observed_path,'m3_path':src.m3_path,'feature_path':src.feature_path,'frozen_base_path':str(base_path),'protected_print_path':str(protected_path),'shape':str(observed.shape),'M3_pixels':int(m3.sum()),'SAFE_REMOVE_pixels':int(safe_remove.sum()),'M5_REMOVE_pixels':int(m5.sum()),'outside_M3_drift':int(np.sum(out[~m3]!=base[~m3])),'M3_sha256':array_hash(m3.astype(np.uint8)),'SAFE_REMOVE_sha256':array_hash(safe_remove.astype(np.uint8)),'M5_REMOVE_sha256':array_hash(m5.astype(np.uint8)),'corrected_sha256':array_hash(out),'notebook51_reference_available':reference_available,'notebook51_corrected_xor_pixels':ref_diff,'notebook51_corrected_exact':exact_reference,'policy_id':EXPECTED_POLICY_ID})
    print('Built',page,'M3',int(m3.sum()),'SAFE_REMOVE',int(safe_remove.sum()),'M5_REMOVE',int(m5.sum()),'N51 exact',exact_reference,'diff',ref_diff)
page_df=pd.DataFrame(page_rows); decisions_df=pd.DataFrame(decision_rows); region_df=pd.DataFrame(region_audit_rows)
page_df.to_csv(TABLES/'operational_page_inventory.csv',index=False); decisions_df.to_parquet(TABLES/'all_operational_safe_remove_decisions.parquet',index=False); region_df.to_csv(TABLES/'regional_reconstruction_inventory.csv',index=False)
require(len(page_df)==len(sources),'Not every input page completed.')
require(page_df.notebook51_reference_available.all(),'Notebook 51R2 corrected-page reference missing for at least one page.')
require(page_df.notebook51_corrected_exact.all(),f'Exact Notebook 51R2 corrected-page replay failed: {page_df.loc[~page_df.notebook51_corrected_exact,["page","notebook51_corrected_xor_pixels"]].to_dict("records")}')


In [ ]:
# ============================================================
# TEMPORARY DIAGNOSTIC CELL
# Localise Notebook53R2 versus Notebook51R2 corrected-page
# differences without changing reconstruction or outputs.
#
# Run after the main page-build cell has completed its loop,
# even if its final exact-equivalence assertion failed.
#
# Requires existing variables:
#   N51
#   PAGES
#   TABLES
#   n51_candidates
# ============================================================

from pathlib import Path
import cv2
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Preconditions
# ------------------------------------------------------------

required_globals = [
    'N51',
    'PAGES',
    'TABLES',
    'n51_candidates',
]

missing_globals = [
    name for name in required_globals
    if name not in globals()
]

if missing_globals:
    raise RuntimeError(
        'Run the Notebook53R2 setup and page-build cell first. '
        f'Missing variables: {missing_globals}'
    )

N51 = Path(N51)
PAGES = Path(PAGES)
TABLES = Path(TABLES)

diagnostic_root = Path(
    globals().get(
        'AUDIT',
        TABLES.parent / 'audit'
    )
) / 'notebook51_difference_localisation'

diagnostic_root.mkdir(
    parents=True,
    exist_ok=True
)

print('=' * 80)
print('NOTEBOOK53R2 / NOTEBOOK51R2 DIFFERENCE LOCALISATION')
print('=' * 80)

print('Notebook51R2:', N51)
print('Notebook53R2 pages:', PAGES)
print('Diagnostic output:', diagnostic_root)


# ------------------------------------------------------------
# 2. Normalise the Notebook51R2 candidate inventory
# ------------------------------------------------------------

inventory = n51_candidates.copy()

required_columns = {
    'page',
    'arm',
    'restorer',
    'candidate',
}

missing_columns = required_columns - set(inventory.columns)

if missing_columns:
    raise RuntimeError(
        'Notebook51R2 candidate inventory lacks columns: '
        f'{sorted(missing_columns)}'
    )

for column in [
    'page',
    'arm',
    'restorer',
    'candidate',
]:
    inventory[column] = (
        inventory[column]
        .astype(str)
        .str.strip()
    )

golden = inventory[
    (inventory.arm == 'M5_REMOVE')
    &
    (inventory.restorer == 'R2_telea')
].copy()

if golden.empty:
    raise RuntimeError(
        'No Notebook51R2 M5_REMOVE / R2_telea rows found.'
    )

if golden.page.duplicated().any():
    duplicates = sorted(
        golden.loc[
            golden.page.duplicated(keep=False),
            'page'
        ].unique()
    )

    raise RuntimeError(
        'Duplicate Notebook51R2 M5_REMOVE/R2_telea '
        f'candidate rows for pages: {duplicates}'
    )

print(
    'Notebook51R2 M5_REMOVE/R2_telea references:',
    len(golden)
)


# ------------------------------------------------------------
# 3. Load the operational page inventory if available
# ------------------------------------------------------------

page_inventory_path = (
    TABLES /
    'operational_page_inventory.csv'
)

if page_inventory_path.exists():
    operational_inventory = pd.read_csv(
        page_inventory_path,
        dtype={'page': str},
    )
else:
    operational_inventory = pd.DataFrame()

print(
    'Operational inventory available:',
    page_inventory_path.exists()
)


# ------------------------------------------------------------
# 4. Helpers
# ------------------------------------------------------------

def load_gray(path, label):
    path = Path(path)

    if not path.exists():
        raise RuntimeError(
            f'Missing {label}: {path}'
        )

    image = cv2.imread(
        str(path),
        cv2.IMREAD_GRAYSCALE,
    )

    if image is None:
        raise RuntimeError(
            f'Unreadable {label}: {path}'
        )

    return image


def load_bool(path, label):
    return load_gray(path, label) > 0


def resolve_reference(page):
    row = golden[
        golden.page == str(page)
    ]

    if len(row) != 1:
        raise RuntimeError(
            f'{page}: expected one Notebook51R2 '
            f'M5_REMOVE/R2_telea row; found {len(row)}.'
        )

    recorded = Path(
        row.iloc[0]['candidate']
    )

    if recorded.exists():
        return recorded, row.iloc[0]

    fallback = (
        N51 /
        '03_candidates' /
        str(page) /
        recorded.name
    )

    if fallback.exists():
        return fallback, row.iloc[0]

    # Diagnostic fallback only. Require a unique exact filename
    # beneath the resolved Notebook51R2 run.
    hits = sorted(
        path for path in N51.rglob(recorded.name)
        if (
            path.is_file()
            and path.parent.name == str(page)
        )
    )

    if len(hits) != 1:
        raise RuntimeError(
            f'{page}: could not resolve a unique Notebook51R2 '
            f'reference.\n'
            f'Recorded: {recorded}\n'
            f'Fallback: {fallback}\n'
            f'Run-wide exact-name hits: {hits}'
        )

    return hits[0], row.iloc[0]


def write_binary(path, mask):
    ok = cv2.imwrite(
        str(path),
        mask.astype(np.uint8) * 255,
    )

    if not ok:
        raise RuntimeError(
            f'Could not write diagnostic image: {path}'
        )


def write_intensity(path, values):
    ok = cv2.imwrite(
        str(path),
        values.astype(np.uint8),
    )

    if not ok:
        raise RuntimeError(
            f'Could not write diagnostic image: {path}'
        )


# ------------------------------------------------------------
# 5. Audit every completed page
# ------------------------------------------------------------

summary_rows = []
coordinate_rows = []

page_dirs = sorted(
    path for path in PAGES.iterdir()
    if path.is_dir()
)

if not page_dirs:
    raise RuntimeError(
        f'No completed page directories found under {PAGES}'
    )

for page_dir in page_dirs:

    page = page_dir.name

    required_page_files = {
        'observed':
            page_dir / 'observed.png',

        'm3':
            page_dir / 'm3.png',

        'safe_remove':
            page_dir / 'safe_remove.png',

        'm5':
            page_dir / 'm5_remove.png',

        'base':
            page_dir / 'frozen_v7_8_base.png',

        'protected':
            page_dir / 'protected_print.png',

        'built':
            page_dir / 'corrected.png',
    }

    missing = [
        name
        for name, path in required_page_files.items()
        if not path.exists()
    ]

    if missing:
        print(
            f'SKIP {page}: missing page artefacts {missing}'
        )
        continue

    observed = load_gray(
        required_page_files['observed'],
        f'{page} observed',
    )

    m3 = load_bool(
        required_page_files['m3'],
        f'{page} M3',
    )

    safe_remove = load_bool(
        required_page_files['safe_remove'],
        f'{page} SAFE_REMOVE',
    )

    m5 = load_bool(
        required_page_files['m5'],
        f'{page} M5_REMOVE',
    )

    base = load_gray(
        required_page_files['base'],
        f'{page} frozen V7.8 base',
    )

    protected = load_bool(
        required_page_files['protected'],
        f'{page} protected print',
    )

    built = load_gray(
        required_page_files['built'],
        f'{page} Notebook53R2 corrected page',
    )

    reference_path, reference_row = resolve_reference(
        page
    )

    reference = load_gray(
        reference_path,
        f'{page} Notebook51R2 reference',
    )

    arrays = {
        'observed': observed,
        'm3': m3,
        'safe_remove': safe_remove,
        'm5': m5,
        'base': base,
        'protected': protected,
        'built': built,
        'reference': reference,
    }

    shapes = {
        name: value.shape
        for name, value in arrays.items()
    }

    if len(set(shapes.values())) != 1:
        raise RuntimeError(
            f'{page}: shape mismatch: {shapes}'
        )

    # --------------------------------------------------------
    # Central difference population
    # --------------------------------------------------------

    different = built != reference
    diff_pixels = int(different.sum())

    signed_delta = (
        built.astype(np.int16)
        -
        reference.astype(np.int16)
    )

    absolute_delta = np.abs(
        signed_delta
    ).astype(np.uint8)

    # --------------------------------------------------------
    # Partition the difference coordinates
    # --------------------------------------------------------

    inside_m3 = different & m3
    outside_m3 = different & ~m3

    inside_m5 = different & m5
    inside_safe_remove = different & safe_remove

    inside_protected = different & protected
    inside_unprotected = different & ~protected

    m5_protected = different & m5 & protected
    m5_unprotected = different & m5 & ~protected

    safe_protected = (
        different
        & safe_remove
        & protected
    )

    safe_unprotected = (
        different
        & safe_remove
        & ~protected
    )

    outside_m3_protected = (
        different
        & ~m3
        & protected
    )

    outside_m3_unprotected = (
        different
        & ~m3
        & ~protected
    )

    # --------------------------------------------------------
    # Determine whether either output inherited an existing
    # value from observed or frozen base at mismatch pixels.
    # --------------------------------------------------------

    built_equals_observed = (
        different
        & (built == observed)
    )

    reference_equals_observed = (
        different
        & (reference == observed)
    )

    built_equals_base = (
        different
        & (built == base)
    )

    reference_equals_base = (
        different
        & (reference == base)
    )

    built_observed_ref_base = (
        different
        & (built == observed)
        & (reference == base)
    )

    built_base_ref_observed = (
        different
        & (built == base)
        & (reference == observed)
    )

    # A mismatch in which both results are independently
    # reconstructed values rather than direct observed/base copies.
    neither_matches_observed_or_base = (
        different
        & (built != observed)
        & (built != base)
        & (reference != observed)
        & (reference != base)
    )

    # --------------------------------------------------------
    # Basic mask partition integrity
    # --------------------------------------------------------

    partition_ok = bool(
        np.array_equal(
            m3,
            safe_remove | m5,
        )
        and not np.any(
            safe_remove & m5
        )
    )

    # --------------------------------------------------------
    # Write page diagnostics
    # --------------------------------------------------------

    page_diag = (
        diagnostic_root /
        page
    )

    page_diag.mkdir(
        parents=True,
        exist_ok=True,
    )

    write_binary(
        page_diag / 'difference_mask.png',
        different,
    )

    write_intensity(
        page_diag / 'absolute_difference.png',
        absolute_delta,
    )

    write_binary(
        page_diag / 'difference_inside_m5.png',
        inside_m5,
    )

    write_binary(
        page_diag / 'difference_inside_safe_remove.png',
        inside_safe_remove,
    )

    write_binary(
        page_diag / 'difference_inside_protected_print.png',
        inside_protected,
    )

    write_binary(
        page_diag / 'difference_outside_m3.png',
        outside_m3,
    )

    # Magnified locator for very small mismatch populations.
    locator = cv2.dilate(
        different.astype(np.uint8),
        np.ones((7, 7), np.uint8),
    ) > 0

    write_binary(
        page_diag / 'difference_locator_7x7.png',
        locator,
    )

    # --------------------------------------------------------
    # Pixel-coordinate table
    # --------------------------------------------------------

    ys, xs = np.where(different)

    for y, x in zip(ys, xs):

        coordinate_rows.append({
            'page':
                page,

            'y':
                int(y),

            'x':
                int(x),

            'built_value':
                int(built[y, x]),

            'reference_value':
                int(reference[y, x]),

            'signed_difference':
                int(signed_delta[y, x]),

            'absolute_difference':
                int(abs(signed_delta[y, x])),

            'observed_value':
                int(observed[y, x]),

            'frozen_base_value':
                int(base[y, x]),

            'inside_M3':
                bool(m3[y, x]),

            'inside_SAFE_REMOVE':
                bool(safe_remove[y, x]),

            'inside_M5_REMOVE':
                bool(m5[y, x]),

            'inside_PROTECTED_PRINT':
                bool(protected[y, x]),

            'built_equals_observed':
                bool(
                    built[y, x]
                    == observed[y, x]
                ),

            'reference_equals_observed':
                bool(
                    reference[y, x]
                    == observed[y, x]
                ),

            'built_equals_frozen_base':
                bool(
                    built[y, x]
                    == base[y, x]
                ),

            'reference_equals_frozen_base':
                bool(
                    reference[y, x]
                    == base[y, x]
                ),

            'reference_path':
                str(reference_path),
        })

    # --------------------------------------------------------
    # Page summary
    # --------------------------------------------------------

    summary_rows.append({
        'page':
            page,

        'different_pixels':
            diff_pixels,

        'max_absolute_difference':
            int(
                absolute_delta.max()
            ),

        'mean_absolute_difference_on_mismatch':
             (
                float(
                    absolute_delta[different].mean()
                )
                if diff_pixels
                else 0.0
            ),

        'mask_partition_ok':
            partition_ok,

        'difference_inside_M3':
            int(inside_m3.sum()),

        'difference_outside_M3':
            int(outside_m3.sum()),

        'difference_inside_M5_REMOVE':
            int(inside_m5.sum()),

        'difference_inside_SAFE_REMOVE':
            int(inside_safe_remove.sum()),

        'difference_inside_PROTECTED_PRINT':
            int(inside_protected.sum()),

        'difference_inside_unprotected':
            int(inside_unprotected.sum()),

       'difference_M5_and_PROTECTED_PRINT':
            int(m5_protected.sum()),

        'difference_M5_and_not_PROTECTED_PRINT':
            int(m5_unprotected.sum()),

        'difference_SAFE_REMOVE_and_PROTECTED_PRINT':
            int(safe_protected.sum()),

        'difference_SAFE_REMOVE_and_not_PROTECTED_PRINT':
            int(safe_unprotected.sum()),

        'difference_outside_M3_and_PROTECTED_PRINT':
            int(outside_m3_protected.sum()),

        'difference_outside_M3_and_not_PROTECTED_PRINT':
            int(outside_m3_unprotected.sum()),

        'built_equals_observed_at_difference':
            int(built_equals_observed.sum()),

        'reference_equals_observed_at_difference':
            int(reference_equals_observed.sum()),

        'built_equals_base_at_difference':
            int(built_equals_base.sum()),

        'reference_equals_base_at_difference':
            int(reference_equals_base.sum()),

        'built_observed_reference_base':
            int(built_observed_ref_base.sum()),

        'built_base_reference_observed':
            int(built_base_ref_observed.sum()),

        'neither_output_matches_observed_or_base':
            int(
                neither_matches_observed_or_base.sum()
            ),

        'M3_pixels':
            int(m3.sum()),

        'SAFE_REMOVE_pixels':
            int(safe_remove.sum()),

        'M5_REMOVE_pixels':
            int(m5.sum()),

        'PROTECTED_PRINT_pixels':
            int(protected.sum()),

        'reference_path':
            str(reference_path),
    })

    print(
        f'{page}: '
        f'diff={diff_pixels}, '
        f'M5={int(inside_m5.sum())}, '
        f'SAFE_REMOVE={int(inside_safe_remove.sum())}, '
        f'PROTECTED={int(inside_protected.sum())}, '
        f'outside_M3={int(outside_m3.sum())}'
    )


# ------------------------------------------------------------
# 6. Write consolidated tables
# ------------------------------------------------------------

summary_df = pd.DataFrame(
    summary_rows
).sort_values(
    [
        'different_pixels',
        'page',
    ],
    ascending=[
        False,
        True,
    ],
).reset_index(drop=True)

coordinate_df = pd.DataFrame(
    coordinate_rows
)

summary_path = (
    diagnostic_root /
    'difference_localisation_by_page.csv'
)

coordinate_path = (
    diagnostic_root /
    'difference_coordinates.csv'
)

summary_df.to_csv(
    summary_path,
    index=False,
)

coordinate_df.to_csv(
    coordinate_path,
    index=False,
)


# ------------------------------------------------------------
# 7. Print concise interpretation table
# ------------------------------------------------------------

display_columns = [
    'page',
    'different_pixels',
    'max_absolute_difference',
    'difference_inside_M5_REMOVE',
    'difference_inside_SAFE_REMOVE',
    'difference_inside_PROTECTED_PRINT',
    'difference_outside_M3',
    'built_equals_observed_at_difference',
    'reference_equals_observed_at_difference',
    'built_equals_base_at_difference',
    'reference_equals_base_at_difference',
    'neither_output_matches_observed_or_base',
]

print('\n' + '=' * 80)
print('PAGES WITH NON-ZERO DIFFERENCES')
print('=' * 80)

nonzero = summary_df[
    summary_df.different_pixels > 0
]

if nonzero.empty:
    print(
        'All Notebook53R2 pages are now exact.'
    )
else:
    display(
        nonzero[
            display_columns
        ]
    )


# ------------------------------------------------------------
# 8. Aggregate mismatch classification
# ------------------------------------------------------------

aggregate_columns = [
    'different_pixels',
    'difference_inside_M3',
    'difference_outside_M3',
    'difference_inside_M5_REMOVE',
    'difference_inside_SAFE_REMOVE',
    'difference_inside_PROTECTED_PRINT',
    'difference_M5_and_PROTECTED_PRINT',
    'difference_M5_and_not_PROTECTED_PRINT',
    'difference_SAFE_REMOVE_and_PROTECTED_PRINT',
    'difference_SAFE_REMOVE_and_not_PROTECTED_PRINT',
    'built_equals_observed_at_difference',
    'reference_equals_observed_at_difference',
    'built_equals_base_at_difference',
    'reference_equals_base_at_difference',
    'neither_output_matches_observed_or_base',
]

aggregate = {
    column: int(
        summary_df[column].sum()
    )
    for column in aggregate_columns
}

print('\n' + '=' * 80)
print('AGGREGATE CLASSIFICATION')
print('=' * 80)

for key, value in aggregate.items():
    print(f'{key}: {value}')


# ------------------------------------------------------------
# 9. Fail only for invalid masks/reference inputs
#
# Genuine output differences are reported, not raised here.
# ------------------------------------------------------------

invalid_partitions = summary_df[
    ~summary_df.mask_partition_ok
]

if len(invalid_partitions):
    raise RuntimeError(
        'Mask partition failure on pages: '
        f'{invalid_partitions.page.tolist()}'
    )

print('\nDiagnostic tables:')
print(' ', summary_path)
print(' ', coordinate_path)

print('\nPer-page diagnostic images:')
print(' ', diagnostic_root)

print(
    '\nThis cell did not modify build_output_exact(), '
    'Notebook53R2 outputs, or Notebook51R2 references.'
)

In [ ]:
# ============================================================
# TEMPORARY DIAGNOSTIC
# Test whether Notebook53R2 / Notebook51R2 differences occur
# at M5 pixels owned by multiple Notebook45 regional M3 masks.
# ============================================================

overlap_rows = []

for page_dir in sorted(
    p for p in PAGES.iterdir()
    if p.is_dir()
):

    page = page_dir.name

    built_path = page_dir / 'corrected.png'
    m5_path = page_dir / 'm5_remove.png'

    if not built_path.exists() or not m5_path.exists():
        continue

    built = cv2.imread(
        str(built_path),
        cv2.IMREAD_GRAYSCALE
    )

    m5 = (
        cv2.imread(
            str(m5_path),
            cv2.IMREAD_GRAYSCALE
        ) > 0
    )

    # -----------------------------------------------
    # Resolve Notebook51R2 reference
    # -----------------------------------------------

    ref_row = n51_candidates[
        (n51_candidates.page.astype(str) == page)
        &
        (n51_candidates.arm == 'M5_REMOVE')
        &
        (n51_candidates.restorer == 'R2_telea')
    ]

    require(
        len(ref_row) == 1,
        f'{page}: expected exactly one '
        f'Notebook51R2 M5_REMOVE/R2_telea row.'
    )

    ref = Path(
        ref_row.iloc[0]['candidate']
    )

    if not ref.exists():

        fallback = (
            N51 /
            '03_candidates' /
            page /
            ref.name
        )

        require(
            fallback.exists(),
            f'{page}: reference not found.'
        )

        ref = fallback

    reference = cv2.imread(
        str(ref),
        cv2.IMREAD_GRAYSCALE
    )

    require(
        reference is not None
        and reference.shape == built.shape,
        f'{page}: invalid reference.'
    )

    different = built != reference

    # -----------------------------------------------
    # Load every Notebook45 regional M3
    # -----------------------------------------------

    pg = regions[
        regions.page.astype(str) == page
    ].copy()

    require(
        len(pg) > 0,
        f'{page}: no Notebook45 regions.'
    )

    ownership = np.zeros(
        built.shape,
        dtype=np.uint16
    )

    region_masks = {}

    for _, r in pg.iterrows():

        region_id = str(r.region_id)

        mp = (
            N45 /
            'masks' /
            page /
            region_id.replace(':', '_') /
            'M3.png'
        )

        rm = cv2.imread(
            str(mp),
            cv2.IMREAD_GRAYSCALE
        )

        require(
            rm is not None,
            f'{page}: missing regional M3: {mp}'
        )

        rm = rm > 0

        require(
            rm.shape == built.shape,
            f'{page}: regional M3 shape mismatch.'
        )

        region_masks[region_id] = rm

        ownership += rm.astype(np.uint16)

    # -----------------------------------------------
    # Classify the differing M5 pixels
    # -----------------------------------------------

    diff_m5 = different & m5

    single_owner = diff_m5 & (ownership == 1)
    multi_owner = diff_m5 & (ownership > 1)
    zero_owner = diff_m5 & (ownership == 0)

    ys, xs = np.where(diff_m5)

    for y, x in zip(ys, xs):

        owners = [
            rid
            for rid, rm in region_masks.items()
            if rm[y, x]
        ]

        overlap_rows.append({
            'page': page,
            'y': int(y),
            'x': int(x),

            'built':
                int(built[y, x]),

            'reference':
                int(reference[y, x]),

            'absolute_difference':
                int(
                    abs(
                        int(built[y, x])
                        -
                        int(reference[y, x])
                    )
                ),

            'regional_owner_count':
                int(ownership[y, x]),

            'regional_owners':
                '|'.join(owners),
        })

    print(
        f'{page}: '
        f'diff_M5={int(diff_m5.sum())}, '
        f'single_owner={int(single_owner.sum())}, '
        f'multi_owner={int(multi_owner.sum())}, '
        f'zero_owner={int(zero_owner.sum())}'
    )


overlap_df = pd.DataFrame(overlap_rows)

overlap_path = (
    AUDIT /
    'm5_difference_regional_ownership.csv'
)

overlap_df.to_csv(
    overlap_path,
    index=False
)

print('\nSaved:')
print(overlap_path)

if len(overlap_df):

    print('\nRegional-owner distribution:')

    display(
        overlap_df
        .groupby(
            'regional_owner_count',
            as_index=False
        )
        .agg(
            pixels=('page', 'size'),
            pages=('page', 'nunique'),
        )
    )

    print('\nDifferences with multiple owners:')

    display(
        overlap_df[
            overlap_df.regional_owner_count > 1
        ].head(100)
    )

In [ ]:
# ============================================================
# TEMPORARY DIAGNOSTIC
# OpenCV/runtime identity and residual-value audit
# ============================================================

from pathlib import Path
import json
import platform
import sys

import cv2
import numpy as np
import pandas as pd


print('=' * 80)
print('CURRENT RUNTIME')
print('=' * 80)

print('Python:')
print(sys.version)

print('\nPlatform:')
print(platform.platform())

print('\nOpenCV:')
print(cv2.__version__)

print('\nOpenCV build information:')
print(cv2.getBuildInformation())


# ------------------------------------------------------------
# 1. Inspect Notebook51R2 environment records
# ------------------------------------------------------------

environment_candidates = [
    N51 / 'software_environment.json',
    N51 / '07_audit' / 'software_environment.json',
    N51 / 'environment.json',
    N51 / '07_audit' / 'environment.json',
    N51 / 'run_manifest.json',
    N51 / 'experiment_manifest.json',
]

print('\n' + '=' * 80)
print('NOTEBOOK51R2 ENVIRONMENT RECORDS')
print('=' * 80)

found_environment = False

for path in environment_candidates:

    if not path.exists():
        continue

    found_environment = True

    print('\nFile:')
    print(path)

    try:
        value = json.loads(
            path.read_text()
        )

        print(
            json.dumps(
                value,
                indent=2,
                default=str,
            )
        )

    except Exception as error:

        print(
            'Could not parse as JSON:',
            repr(error),
        )


if not found_environment:

    print(
        '\nNo conventional Notebook51R2 environment '
        'record was found.'
    )

    print(
        'Searching Notebook51R2 for files containing '
        '"environment", "software" or "version".'
    )

    environment_search = sorted(
        path
        for path in N51.rglob('*')
        if (
            path.is_file()
            and any(
                token in path.name.lower()
                for token in (
                    'environment',
                    'software',
                    'version',
                    'package',
                    'dependency',
                )
            )
        )
    )

    for path in environment_search:
        print(' ', path)


# ------------------------------------------------------------
# 2. Localise magnitude and sign of actual page differences
# ------------------------------------------------------------

difference_rows = []

golden = n51_candidates[
    (n51_candidates.arm.astype(str) == 'M5_REMOVE')
    &
    (
        n51_candidates.restorer.astype(str)
        == 'R2_telea'
    )
].copy()

require(
    not golden.empty,
    'No Notebook51R2 M5_REMOVE/R2_telea '
    'references were found.'
)

for page_dir in sorted(
    path
    for path in PAGES.iterdir()
    if path.is_dir()
):

    page = page_dir.name

    built_path = (
        page_dir /
        'corrected.png'
    )

    if not built_path.exists():
        continue

    row = golden[
        golden.page.astype(str) == page
    ]

    require(
        len(row) == 1,
        f'{page}: expected one golden-master row; '
        f'found {len(row)}.'
    )

    ref_path = Path(
        row.iloc[0]['candidate']
    )

    if not ref_path.exists():

        ref_path = (
            N51 /
            '03_candidates' /
            page /
            Path(
                row.iloc[0]['candidate']
            ).name
        )

    require(
        ref_path.exists(),
        f'{page}: reference is missing: {ref_path}'
    )

    built = cv2.imread(
        str(built_path),
        cv2.IMREAD_GRAYSCALE,
    )

    reference = cv2.imread(
        str(ref_path),
        cv2.IMREAD_GRAYSCALE,
    )

    require(
        built is not None
        and reference is not None
        and built.shape == reference.shape,
        f'{page}: invalid built/reference arrays.'
    )

    ys, xs = np.where(
        built != reference
    )

    for y, x in zip(ys, xs):

        b = int(built[y, x])
        r = int(reference[y, x])

        difference_rows.append({
            'page':
                page,

            'y':
                int(y),

            'x':
                int(x),

            'notebook53r2_value':
                b,

            'notebook51r2_value':
                r,

            'signed_difference':
                b - r,

            'absolute_difference':
                abs(b - r),
        })


difference_values = pd.DataFrame(
    difference_rows
)

difference_path = (
    AUDIT /
    'telea_residual_value_differences.csv'
)

difference_values.to_csv(
    difference_path,
    index=False,
)


print('\n' + '=' * 80)
print('RESIDUAL VALUE DIFFERENCES')
print('=' * 80)

print('Differing pixels:', len(difference_values))

if len(difference_values):

    print('\nAbsolute-difference distribution:')

    display(
        difference_values
        .absolute_difference
        .value_counts()
        .sort_index()
        .rename_axis(
            'absolute_difference'
        )
        .reset_index(
            name='pixels'
        )
    )

    print('\nPer-page summary:')

    display(
        difference_values
        .groupby(
            'page',
            as_index=False,
        )
        .agg(
            differing_pixels=(
                'absolute_difference',
                'size',
            ),
            maximum_absolute_difference=(
                'absolute_difference',
                'max',
            ),
            mean_absolute_difference=(
                'absolute_difference',
                'mean',
            ),
            minimum_signed_difference=(
                'signed_difference',
                'min',
            ),
            maximum_signed_difference=(
                'signed_difference',
                'max',
            ),
        )
        .sort_values(
            [
                'differing_pixels',
                'page',
            ],
            ascending=[
                False,
                True,
            ],
        )
    )

    print('\nSigned-difference distribution:')

    display(
        difference_values
        .signed_difference
        .value_counts()
        .sort_index()
        .rename_axis(
            'signed_difference'
        )
        .reset_index(
            name='pixels'
        )
    )

else:

    print(
        'All currently completed pages are exact.'
    )


print('\nSaved:')
print(difference_path)

In [ ]:
# ============================================================
# TEMPORARY EXPERIMENT CELL
#
# Identify which Telea context reproduces Notebook 51R2:
#
#   A. full-page Telea for each Notebook45 regional M5
#   B. canonical connected-component crops, margin 48
#   C. one regional-mask bounding-box crop, margin 48
#
# This cell DOES NOT modify existing Notebook53R2 outputs.
# It writes a separate diagnostic package under AUDIT.
#
# Expected existing variables:
#   N51
#   N45
#   PAGES
#   AUDIT
#   regions
#   n51_candidates
# ============================================================

from pathlib import Path
import json
import hashlib

import cv2
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

TELEA_RADIUS = 3
CROP_MARGIN = 48
MIN_COMPONENT_AREA = 8

PAGES = Path(PAGES)
AUDIT = Path(AUDIT)
N45 = Path(N45)
N51 = Path(N51)

EXPERIMENT_ROOT = (
    AUDIT /
    'telea_context_experiment'
)

EXPERIMENT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

EXPERIMENT_IMAGES = (
    EXPERIMENT_ROOT /
    'pages'
)

EXPERIMENT_IMAGES.mkdir(
    parents=True,
    exist_ok=True,
)

print('=' * 80)
print('TELEA CONTEXT EQUIVALENCE EXPERIMENT')
print('=' * 80)

print('Notebook45:', N45)
print('Notebook51R2:', N51)
print('Notebook53R2 pages:', PAGES)
print('Output:', EXPERIMENT_ROOT)
print('Telea radius:', TELEA_RADIUS)
print('Crop margin:', CROP_MARGIN)
print('Minimum connected-component area:', MIN_COMPONENT_AREA)


# ------------------------------------------------------------
# 2. Preconditions
# ------------------------------------------------------------

required_names = [
    'regions',
    'n51_candidates',
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        'Required Notebook53R2 variables are missing: '
        f'{missing_names}'
    )

required_region_columns = {
    'page',
    'region_id',
}

missing_region_columns = (
    required_region_columns
    -
    set(regions.columns)
)

if missing_region_columns:
    raise RuntimeError(
        'Notebook45 region manifest lacks columns: '
        f'{sorted(missing_region_columns)}'
    )

regions_test = regions.copy()

regions_test['page'] = (
    regions_test['page']
    .astype(str)
)

regions_test['region_id'] = (
    regions_test['region_id']
    .astype(str)
)


# ------------------------------------------------------------
# 3. Resolve Notebook51R2 golden-master inventory
# ------------------------------------------------------------

inventory = n51_candidates.copy()

required_candidate_columns = {
    'page',
    'arm',
    'restorer',
    'candidate',
}

missing_candidate_columns = (
    required_candidate_columns
    -
    set(inventory.columns)
)

if missing_candidate_columns:
    raise RuntimeError(
        'Notebook51R2 candidate inventory lacks columns: '
        f'{sorted(missing_candidate_columns)}'
    )

for column in [
    'page',
    'arm',
    'restorer',
    'candidate',
]:
    inventory[column] = (
        inventory[column]
        .astype(str)
        .str.strip()
    )

golden = inventory[
    (inventory.arm == 'M5_REMOVE')
    &
    (inventory.restorer == 'R2_telea')
].copy()

if golden.empty:
    raise RuntimeError(
        'No Notebook51R2 M5_REMOVE/R2_telea '
        'golden-master rows were found.'
    )

if golden.page.duplicated().any():
    duplicate_pages = sorted(
        golden.loc[
            golden.page.duplicated(
                keep=False
            ),
            'page'
        ].unique()
    )

    raise RuntimeError(
        'Duplicate Notebook51R2 golden-master rows: '
        f'{duplicate_pages}'
    )

print(
    'Notebook51R2 M5_REMOVE/R2_telea references:',
    len(golden)
)


# ------------------------------------------------------------
# 4. General helpers
# ------------------------------------------------------------

def require_local(condition, message):
    if not condition:
        raise RuntimeError(message)


def array_sha256(array):
    array = np.ascontiguousarray(array)

    h = hashlib.sha256()
    h.update(str(array.dtype).encode())
    h.update(str(array.shape).encode())
    h.update(array.tobytes())

    return h.hexdigest()


def load_gray(path, description):
    path = Path(path)

    require_local(
        path.exists(),
        f'Missing {description}: {path}'
    )

    image = cv2.imread(
        str(path),
        cv2.IMREAD_GRAYSCALE,
    )

    require_local(
        image is not None,
        f'Unreadable {description}: {path}'
    )

    return image


def load_bool(path, description):
    return load_gray(
        path,
        description,
    ) > 0


def write_gray(path, image):
    path = Path(path)

    ok = cv2.imwrite(
        str(path),
        image,
    )

    require_local(
        ok,
        f'Could not write image: {path}'
    )


def write_mask(path, mask):
    write_gray(
        path,
        mask.astype(np.uint8) * 255,
    )


def resolve_reference(page):
    row = golden[
        golden.page == str(page)
    ]

    require_local(
        len(row) == 1,
        f'{page}: expected one golden-master row; '
        f'found {len(row)}.'
    )

    recorded = Path(
        row.iloc[0]['candidate']
    )

    if recorded.exists():
        return recorded

    fallback = (
        N51 /
        '03_candidates' /
        str(page) /
        recorded.name
    )

    if fallback.exists():
        return fallback

    hits = sorted(
        path
        for path in N51.rglob(recorded.name)
        if (
            path.is_file()
            and path.parent.name == str(page)
        )
    )

    require_local(
        len(hits) == 1,
        f'{page}: could not resolve one Notebook51R2 '
        f'golden master.\n'
        f'Recorded: {recorded}\n'
        f'Direct fallback: {fallback}\n'
        f'Run-wide hits: {hits}'
    )

    return hits[0]


def resolve_region_m3(page, region_id):
    region_folder = (
        str(region_id)
        .replace(':', '_')
    )

    primary = (
        N45 /
        'masks' /
        str(page) /
        region_folder /
        'M3.png'
    )

    if primary.exists():
        return primary

    page_root = (
        N45 /
        'masks' /
        str(page)
    )

    require_local(
        page_root.exists(),
        f'{page}: Notebook45 page-mask directory '
        f'is missing: {page_root}'
    )

    # Diagnostic fallback only. Require one exact M3.png whose
    # parent folder corresponds to the normalised region ID.
    hits = sorted(
        path
        for path in page_root.rglob('M3.png')
        if (
            path.parent.name
            .replace(':', '_')
            == region_folder
        )
    )

    require_local(
        len(hits) == 1,
        f'{page}/{region_id}: expected one regional M3, '
        f'found {len(hits)}: {hits}'
    )

    return hits[0]


# ------------------------------------------------------------
# 5. Telea variant A: full

## 9. Registered-replay equivalence against Notebook 50 semantic maps


In [ ]:
replay_rows=[]
if INPUT_MODE=='REGISTERED_REPLAY':
    for _,r in page_df.iterrows():
        page=r.page; inv=map_inventory[map_inventory.page==page]; p=resolve_unique(inv.iloc[0]['file'],[N50,N50/'maps',N50/'masks'])
        with np.load(p,allow_pickle=False) as z: dm=np.asarray(z['m5_decision_map'])
        saved=(dm==SAFE_REMOVE_CODE)
        built=np.load(MASKS/f'{page}_m5_remove_operational.npz')['SAFE_REMOVE']>0
        # Notebook 50 may have a bottom/right truncated decision canvas. Pad only at origin.
        require(saved.shape[0]<=built.shape[0] and saved.shape[1]<=built.shape[1],f'{page}: saved decision canvas exceeds page.')
        padded=np.zeros(built.shape,bool); padded[:saved.shape[0],:saved.shape[1]]=saved
        xor=int(np.logical_xor(padded,built).sum())
        replay_rows.append({'page':page,'saved_shape':str(saved.shape),'operational_shape':str(built.shape),'safe_remove_xor_pixels':xor,'exact':xor==0,'saved_semantic_sha256':array_hash(saved.astype(np.uint8)),'padded_semantic_sha256':array_hash(padded.astype(np.uint8)),'operational_semantic_sha256':array_hash(built.astype(np.uint8))})
    replay_df=pd.DataFrame(replay_rows); replay_df.to_csv(TABLES/'notebook50_safe_remove_replay.csv',index=False)
    require(replay_df.exact.all(),f'Notebook 50 SAFE_REMOVE replay failure on: {replay_df.loc[~replay_df.exact,"page"].tolist()}')
    print('Exact SAFE_REMOVE replay pages:',len(replay_df))
else:
    replay_df=pd.DataFrame(columns=['page','exact']); replay_df.to_csv(TABLES/'notebook50_safe_remove_replay.csv',index=False)


## 10. Fail-closed V8.8B verdict


In [ ]:
gates={
 'notebook52_policy_pass':verdict52.get('verdict')=='PASS_M5_REMOVE_FREEZE_READY_FOR_OPERATIONAL_BUILDER',
 'policy_identity_exact':policy.get('policy_id')==EXPECTED_POLICY_ID,
 'remove_only':policy.get('safe_protect_runtime_enabled') is False,
 'all_pages_completed':len(page_df)==len(sources),
 'safe_remove_subset_m3':bool((page_df.SAFE_REMOVE_pixels<=page_df.M3_pixels).all()),
 'notebook51_corrected_page_exact':bool(page_df.notebook51_corrected_exact.all()),
 'notebook50_replay_exact':bool(replay_df.exact.all()) if INPUT_MODE=='REGISTERED_REPLAY' and len(replay_df) else None,
}
required_gate_names=['notebook52_policy_pass','policy_identity_exact','remove_only','all_pages_completed','safe_remove_subset_m3','notebook51_corrected_page_exact']
if INPUT_MODE=='REGISTERED_REPLAY': required_gate_names.append('notebook50_replay_exact')
first_failed=next((k for k in required_gate_names if gates.get(k) is not True),None)
verdict='PASS_REGISTERED_REPLAY_READY_FOR_NOTEBOOK54_EQUIVALENCE' if first_failed is None and INPUT_MODE=='REGISTERED_REPLAY' else ('PASS_OPERATIONAL_BUILD_READY_FOR_NOTEBOOK54_EQUIVALENCE' if first_failed is None else 'FAIL_V8_8B_OPERATIONAL_BUILDER')
record={'created_utc':datetime.now(timezone.utc).isoformat(),'verdict':verdict,'first_failed_gate':first_failed,'gates':gates,'mode':INPUT_MODE,'policy_id':EXPECTED_POLICY_ID,'notebook52_run':str(N52),'notebook50_run':str(N50),'pages':len(page_df),'reconstruction_method':RECONSTRUCTION_METHOD,'promotion_authority':False,'meaning':'PASS permits Notebook 54 exact-equivalence testing only. It does not replace Notebook 09 or authorise automatic output.'}
atomic_json(OUT/'notebook53_verdict.json',record); pd.DataFrame([{'gate':k,'passed':v} for k,v in gates.items()]).to_csv(TABLES/'notebook53_gate_results.csv',index=False)
print(json.dumps(record,indent=2))
if first_failed: raise RuntimeError(f'Notebook 53 failed closed at {first_failed}.')


## 11. Identity, source fingerprint and upload package


In [ ]:
environment={'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.8B-notebook53-operational-m5-remove-builder','mode':INPUT_MODE,'policy_id':EXPECTED_POLICY_ID,'notebook52_verdict_sha256':sha256_file(N52/'notebook52_verdict.json'),'policy_manifest_sha256':sha256_file(N52/'policy_bundle'/'policy_manifest.json'),'notebook50_predictions_sha256':sha256_file(pred_path),'predictor_schema_sha256':sha256_file(N50/'inputs'/'predictor_schema.json'),'safe_remove_code':SAFE_REMOVE_CODE,'reconstruction_method':RECONSTRUCTION_METHOD,'inpaint_radius':INPAINT_RADIUS,'notebook51_run':str(N51),'notebook45_run':str(N45),'notebook46_run':str(N46),'composition':'Notebook51R2 frozen V7.8 base + protected print + regional full-page Telea','pages':len(page_df),'page_inventory_sha256':sha256_file(TABLES/'operational_page_inventory.csv'),'decision_table_sha256':sha256_file(TABLES/'all_operational_safe_remove_decisions.parquet'),'retrained':False,'recalibrated':False,'threshold_changed':False,'safe_protect_enabled':False,'clean_truth_used':False,'promotion_authority':False,'software':environment}
identity['implementation_id']=hashlib.sha256(json.dumps({k:identity[k] for k in ['policy_id','policy_manifest_sha256','notebook50_predictions_sha256','predictor_schema_sha256','reconstruction_method','inpaint_radius','composition']},sort_keys=True).encode()).hexdigest()
atomic_json(OUT/'experiment_identity.json',identity); atomic_json(OUT/'software_environment.json',environment)
summary=['NOTEBOOK 53 / V8.8B OPERATIONAL M5_REMOVE BUILDER','',f'Verdict: {verdict}',f'Mode: {INPUT_MODE}',f'Policy ID: {EXPECTED_POLICY_ID}',f'Implementation ID: {identity["implementation_id"]}',f'Pages: {len(page_df)}',f'M3 pixels: {int(page_df.M3_pixels.sum())}',f'SAFE_REMOVE pixels: {int(page_df.SAFE_REMOVE_pixels.sum())}',f'M5_REMOVE pixels: {int(page_df.M5_REMOVE_pixels.sum())}',f'Notebook51 corrected pages exact: {bool(page_df.notebook51_corrected_exact.all())}',f'Notebook51 differing pixels: {int(page_df.notebook51_corrected_xor_pixels.sum())}',f'Notebook 50 replay exact: {gates.get("notebook50_replay_exact")}','','No retraining, recalibration, threshold change or SAFE_PROTECT intervention occurred.','PASS authorises Notebook 54 exact-equivalence testing only.']
(OUT/'executive_summary.txt').write_text(chr(10).join(summary))
upload_files=[OUT/'executive_summary.txt',OUT/'notebook53_verdict.json',OUT/'experiment_identity.json',OUT/'software_environment.json',TABLES/'notebook53_gate_results.csv',TABLES/'operational_page_inventory.csv',TABLES/'notebook50_safe_remove_replay.csv',TABLES/'regional_reconstruction_inventory.csv',AUDIT/'decision_encoding_audit.csv',INPUTS/'resolved_page_inputs.csv']
rows=[]
for src in upload_files:
    require(src.exists(),f'Missing required upload artefact: {src}')
    dst=UPLOAD/src.name; shutil.copy2(src,dst); rows.append({'file':dst.name,'bytes':dst.stat().st_size,'sha256':sha256_file(dst),'source':str(src.relative_to(OUT))})
pd.DataFrame(rows).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('Notebook 53 V8.8B curated review package. Heavy page images, NPZ masks and parquet decisions remain in the durable run. PASS authorises Notebook 54 equivalence testing only.' + chr(10))
zip_path=UPLOAD/'notebook53_v8_8b_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.iterdir()):
        if p.is_file() and p!=zip_path:z.write(p,p.name)
require(zip_path.exists() and zip_path.stat().st_size>0,'Upload bundle creation failed.')
print(chr(10).join(summary)); print('Upload bundle:',zip_path)


## Outputs to inspect first

1. `executive_summary.txt`
2. `notebook53_verdict.json`
3. `tables/notebook50_safe_remove_replay.csv`
4. `tables/operational_page_inventory.csv`
5. `audit/decision_encoding_audit.csv`
6. `pages/<page>/corrected.png`
7. `pages/<page>/safe_remove.png`
8. `pages/<page>/m5_remove.png`
9. `upload/notebook53_v8_8b_upload_bundle.zip`

A registered-replay PASS is the expected first result. It proves the frozen SAFE_REMOVE map can be reproduced in the operational wrapper. Notebook 54 must then compare M3, M5_REMOVE and corrected-page arrays against the V8.7 golden masters before any Notebook 09 shadow use.
